# MDL local-rule induction for ARC-AGI-2 (CPU only)

Transform search + minimum-description-length rule induction with Bayesian model averaging. No internet, no GPU, no pretrained weights. Source: see the linked paper / GitHub repository.

In [ ]:
import os, json
os.makedirs('arcsolver', exist_ok=True)
FILES = {"__init__.py": "", "dsl.py": "\"\"\"Grid-level transforms. Each returns a grid or None when not applicable.\"\"\"\nfrom collections import Counter\n\nimport numpy as np\n\nfrom .grid import background, components, crop, nonbg_bbox, split_cells\n\nTRANSFORMS = []  # (name, fn, cost)\n\n\ndef T(name, cost=1.0):\n    def deco(fn):\n        TRANSFORMS.append((name, fn, cost))\n        return fn\n    return deco\n\n\nT(\"id\", 0.0)(lambda a: a)\nT(\"rot90\")(lambda a: np.rot90(a, 1))\nT(\"rot180\")(lambda a: np.rot90(a, 2))\nT(\"rot270\")(lambda a: np.rot90(a, 3))\nT(\"flipud\")(lambda a: a[::-1])\nT(\"fliplr\")(lambda a: a[:, ::-1])\nT(\"transpose\")(lambda a: a.T)\nT(\"antitranspose\")(lambda a: np.rot90(a, 2).T)\n\n\n@T(\"crop_nonbg\")\ndef _crop_nonbg(a):\n    bb = nonbg_bbox(a, background(a))\n    return None if bb is None else crop(a, bb)\n\n\ndef _crop_obj(select, diag=True, by_color=False):\n    def fn(a):\n        bg = background(a)\n        objs = components(a, bg, diag=diag, by_color=by_color)\n        if not objs:\n            return None\n        o = select(objs, a)\n        return None if o is None else crop(a, o[\"bbox\"])\n    return fn\n\n\ndef _unique_by(key):\n    def sel(objs, a):\n        vals = Counter(key(o, a) for o in objs)\n        cands = [o for o in objs if vals[key(o, a)] == 1]\n        return cands[0] if len(cands) == 1 else None\n    return sel\n\n\ndef _argbest(key, largest=True):\n    def sel(objs, a):\n        ks = [key(o, a) for o in objs]\n        b = max(ks) if largest else min(ks)\n        c = [o for o, k in zip(objs, ks) if k == b]\n        return c[0] if len(c) == 1 else None\n    return sel\n\n\n_bbox_area = lambda o, a: (o[\"bbox\"][1] - o[\"bbox\"][0]) * (o[\"bbox\"][3] - o[\"bbox\"][2])\nfor diag, by_color, tag in ((True, False, \"m\"), (False, True, \"o\")):\n    T(f\"crop_{tag}_largest\", 1.5)(_crop_obj(_argbest(lambda o, a: o[\"size\"]), diag, by_color))\n    T(f\"crop_{tag}_smallest\", 1.5)(_crop_obj(_argbest(lambda o, a: o[\"size\"], False), diag, by_color))\n    T(f\"crop_{tag}_bbox_largest\", 1.5)(_crop_obj(_argbest(_bbox_area), diag, by_color))\n    T(f\"crop_{tag}_unique_color\", 1.5)(_crop_obj(_unique_by(lambda o, a: o[\"color\"]), diag, by_color))\n    T(f\"crop_{tag}_unique_size\", 1.5)(_crop_obj(_unique_by(lambda o, a: o[\"size\"]), diag, by_color))\n    T(f\"crop_{tag}_unique_shape\", 2.0)(_crop_obj(\n        _unique_by(lambda o, a: crop(o[\"mask\"], o[\"bbox\"]).tobytes() + bytes(crop(o[\"mask\"], o[\"bbox\"]).shape)), diag, by_color))\nT(\"crop_m_most_colors\", 1.5)(_crop_obj(_argbest(lambda o, a: len(np.unique(a[o[\"mask\"]]))), True, False))\nT(\"crop_m_densest\", 2.0)(_crop_obj(_argbest(lambda o, a: o[\"size\"] / max(1, _bbox_area(o, a))), True, False))\n\n\ndef _crop_color(rank):\n    def fn(a):\n        bg = background(a)\n        cnt = Counter(a[a != bg].tolist())\n        if len(cnt) <= rank:\n            return None\n        order = [c for c, _ in sorted(cnt.items(), key=lambda t: (t[1], t[0]))]\n        c = order[rank]\n        rows, cols = np.nonzero(a == c)\n        return a[rows.min():rows.max() + 1, cols.min():cols.max() + 1]\n    return fn\n\n\nT(\"crop_rarest_color\", 1.5)(_crop_color(0))\nT(\"crop_2nd_rarest_color\", 2.0)(_crop_color(1))\n\n\ndef _crop_inside_rarest(a):\n    g = _crop_color(0)(a)\n    if g is None or g.shape[0] < 3 or g.shape[1] < 3:\n        return None\n    return g[1:-1, 1:-1]\n\n\nT(\"crop_inside_rarest\", 2.0)(_crop_inside_rarest)\n\nfor k in (2, 3, 4, 5):\n    T(f\"upscale{k}\", 1.0)(lambda a, k=k: np.kron(a, np.ones((k, k), dtype=a.dtype)))\n\n\ndef _downscale(k, mode):\n    def fn(a):\n        h, w = a.shape\n        if h % k or w % k:\n            return None\n        b = a.reshape(h // k, k, w // k, k).transpose(0, 2, 1, 3).reshape(h // k, w // k, k * k)\n        bg = background(a)\n        if mode == \"first\":\n            return b[:, :, 0]\n        out = np.empty((h // k, w // k), dtype=a.dtype)\n        for i in range(h // k):\n            for j in range(w // k):\n                v = b[i, j]\n                nz = v[v != bg]\n                out[i, j] = Counter(nz.tolist()).most_common(1)[0][0] if (mode == \"any\" and len(nz)) else (\n                    Counter(v.tolist()).most_common(1)[0][0] if mode == \"maj\" else bg)\n        return out\n    return fn\n\n\nfor k in (2, 3, 4, 5):\n    T(f\"down{k}_maj\", 1.5)(_downscale(k, \"maj\"))\n    T(f\"down{k}_any\", 1.5)(_downscale(k, \"any\"))\n\nfor n in (1, 2, 3, 4):\n    for m in (1, 2, 3, 4):\n        if n * m > 1:\n            T(f\"tile{n}x{m}\", 1.0)(lambda a, n=n, m=m: np.tile(a, (n, m)))\n\nT(\"mirror_h\", 1.0)(lambda a: np.hstack([a, a[:, ::-1]]))\nT(\"mirror_h2\", 1.0)(lambda a: np.hstack([a[:, ::-1], a]))\nT(\"mirror_v\", 1.0)(lambda a: np.vstack([a, a[::-1]]))\nT(\"mirror_v2\", 1.0)(lambda a: np.vstack([a[::-1], a]))\nT(\"mirror_4\", 1.0)(lambda a: np.vstack([np.hstack([a, a[:, ::-1]]), np.hstack([a[::-1], a[::-1, ::-1]])]))\nT(\"mirror_4b\", 1.0)(lambda a: np.vstack([np.hstack([a[::-1, ::-1], a[::-1]]), np.hstack([a[:, ::-1], a])]))\nT(\"rot_4\", 1.5)(lambda a: np.vstack([np.hstack([a, np.rot90(a, 3)]), np.hstack([np.rot90(a, 1), np.rot90(a, 2)])])\n                if a.shape[0] == a.shape[1] else None)\ndef _self_kron(a):\n    if a.size > 36:\n        return None\n    bg = background(a)\n    h, w = a.shape\n    return np.where(np.kron(a != bg, np.ones((h, w), dtype=bool)), np.tile(a, (h, w)), bg).astype(a.dtype)\n\n\nT(\"self_kron\", 2.0)(_self_kron)\n\n\ndef _parts(a):\n    \"\"\"Sub-grids from separator lines or, failing that, equal halves.\"\"\"\n    s = split_cells(a)\n    out = []\n    if s is not None:\n        _, cells = s\n        flat = [c for row in cells for c in row]\n        if len(flat) >= 2 and all(c.shape == flat[0].shape for c in flat):\n            out.append((\"sep\", flat))\n    h, w = a.shape\n    if w % 2 == 0:\n        out.append((\"lr\", [a[:, : w // 2], a[:, w // 2:]]))\n    if h % 2 == 0:\n        out.append((\"tb\", [a[: h // 2], a[h // 2:]]))\n    return out\n\n\ndef _overlay(kind, which):\n    def fn(a):\n        bg = background(a)\n        for tag, parts in _parts(a):\n            if tag != which:\n                continue\n            st = np.stack(parts)\n            nb = st != bg\n            if kind == \"count\":\n                return nb.sum(0).astype(a.dtype)\n            if kind == \"mask_and\":\n                return nb.all(0).astype(a.dtype)\n            if kind == \"mask_or\":\n                return nb.any(0).astype(a.dtype)\n            if kind == \"mask_xor\":\n                return (nb.sum(0) == 1).astype(a.dtype)\n            if kind == \"bits\":\n                # Each part contributes a bit; rule learner maps codes to colors.\n                return sum((nb[i].astype(int) << i) for i in range(min(len(parts), 3))).astype(a.dtype)\n            if kind in (\"first\", \"last\"):\n                order = range(len(parts)) if kind == \"last\" else range(len(parts) - 1, -1, -1)\n                out = np.full(parts[0].shape, bg, dtype=a.dtype)\n                for i in order:\n                    out = np.where(nb[i], st[i], out)\n                return out\n            if kind in (\"most\", \"least\", \"unique\"):\n                dens = nb.reshape(len(parts), -1).sum(1)\n                if kind == \"unique\":\n                    keys = [p.tobytes() for p in parts]\n                    cnt = Counter(keys)\n                    u = [i for i, k in enumerate(keys) if cnt[k] == 1]\n                    return parts[u[0]] if len(u) == 1 else None\n                b = dens.max() if kind == \"most\" else dens.min()\n                idx = np.flatnonzero(dens == b)\n                return parts[idx[0]] if len(idx) == 1 else None\n        return None\n    return fn\n\n\nfor which in (\"sep\", \"lr\", \"tb\"):\n    for kind in (\"count\", \"bits\", \"mask_and\", \"mask_or\", \"mask_xor\", \"first\", \"last\", \"most\", \"least\", \"unique\"):\n        T(f\"parts_{which}_{kind}\", 1.5)(_overlay(kind, which))\n\n\ndef _cell_summary(a):\n    s = split_cells(a)\n    if s is None:\n        return None\n    sep, cells = s\n    out = np.empty((len(cells), len(cells[0])), dtype=a.dtype)\n    for i, row in enumerate(cells):\n        if len(row) != out.shape[1]:\n            return None\n        for j, c in enumerate(row):\n            v = c[(c != sep)]\n            bgc = background(a)\n            nz = v[v != bgc]\n            out[i, j] = Counter(nz.tolist()).most_common(1)[0][0] if len(nz) else bgc\n    return out\n\n\nT(\"cell_summary\", 1.5)(_cell_summary)\n\n\ndef _remove_separators(a):\n    s = split_cells(a)\n    if s is None:\n        return None\n    _, cells = s\n    try:\n        return np.vstack([np.hstack(row) for row in cells])\n    except ValueError:\n        return None\n\n\nT(\"remove_seps\", 1.5)(_remove_separators)\n\n\ndef _dedupe(a):\n    rows = [0] + [i for i in range(1, a.shape[0]) if not (a[i] == a[i - 1]).all()]\n    b = a[rows]\n    cols = [0] + [j for j in range(1, b.shape[1]) if not (b[:, j] == b[:, j - 1]).all()]\n    return b[:, cols]\n\n\nT(\"dedupe_rc\", 1.5)(_dedupe)\n\n\ndef _drop_bg_lines(a):\n    bg = background(a)\n    rows = [i for i in range(a.shape[0]) if (a[i] != bg).any()]\n    cols = [j for j in range(a.shape[1]) if (a[:, j] != bg).any()]\n    if not rows or not cols:\n        return None\n    return a[rows][:, cols]\n\n\nT(\"drop_bg_lines\", 1.5)(_drop_bg_lines)\n\n\ndef apply(fn, a):\n    try:\n        r = fn(a)\n    except Exception:\n        return None\n    if r is None or r.ndim != 2 or r.size == 0 or r.shape[0] > 30 or r.shape[1] > 30:\n        return None\n    return np.ascontiguousarray(r)\n", "features.py": "\"\"\"Per-cell feature extraction.\n\nEvery feature is an integer array with the same shape as the grid. Values are\nsmall non-negative integers (colors are 0..9; \"none/outside\" is encoded as 10)\nso feature tuples can be packed into a single int64 key.\n\"\"\"\nimport numpy as np\nfrom scipy import ndimage\n\nfrom .grid import background, enclosed, label_map\n\nNONE = 10  # \"outside grid\" / \"no such cell\"\nDIRS4 = {\"u\": (-1, 0), \"d\": (1, 0), \"l\": (0, -1), \"r\": (0, 1)}\nDIRS8 = dict(DIRS4, ul=(-1, -1), ur=(-1, 1), dl=(1, -1), dr=(1, 1))\n\n# Features whose value is a color (can be used as copy source for targets).\nCOLOR_FEATURES = set()\n\n\ndef _shift(a, dr, dc, fill=NONE):\n    h, w = a.shape\n    out = np.full((h, w), fill, dtype=np.int16)\n    rs = slice(max(0, -dr), min(h, h - dr))\n    rd = slice(max(0, dr), min(h, h + dr))\n    cs = slice(max(0, -dc), min(w, w - dc))\n    cd = slice(max(0, dc), min(w, w + dc))\n    out[rs, cs] = a[rd, cd]\n    return out\n\n\ndef _ray(a, bg, dr, dc):\n    \"\"\"First non-background color seen walking from each cell in (dr, dc).\"\"\"\n    h, w = a.shape\n    out = np.full((h, w), NONE, dtype=np.int16)\n    rows = range(h) if dr <= 0 else range(h - 1, -1, -1)\n    cols = range(w) if dc <= 0 else range(w - 1, -1, -1)\n    # Dynamic programming over the walk direction.\n    for r in rows:\n        for c in cols:\n            rr, cc = r + dr, c + dc\n            if 0 <= rr < h and 0 <= cc < w:\n                out[r, c] = a[rr, cc] if a[rr, cc] != bg else out[rr, cc]\n    return out\n\n\ndef _line_color(a, bg, axis):\n    \"\"\"Most common non-bg color along the row (axis=1) or column (axis=0).\"\"\"\n    h, w = a.shape\n    out = np.full((h, w), NONE, dtype=np.int16)\n    n = h if axis == 1 else w\n    for i in range(n):\n        line = a[i] if axis == 1 else a[:, i]\n        vals = line[line != bg]\n        if len(vals):\n            c = np.bincount(vals, minlength=10).argmax()\n            if axis == 1:\n                out[i, :] = c\n            else:\n                out[:, i] = c\n    return out\n\n\ndef cell_features(a: np.ndarray, bg: int = None) -> dict:\n    a = a.astype(np.int16)\n    if bg is None:\n        bg = background(a)\n    h, w = a.shape\n    F = {}\n\n    def color_feat(name, arr):\n        F[name] = arr\n        COLOR_FEATURES.add(name)\n\n    color_feat(\"c\", a)\n    for k, (dr, dc) in DIRS8.items():\n        color_feat(\"n_\" + k, _shift(a, dr, dc))\n    for k, (dr, dc) in DIRS8.items():\n        color_feat(\"ray_\" + k, _ray(a, bg, dr, dc))\n    color_feat(\"mir_h\", a[:, ::-1].copy())\n    color_feat(\"mir_v\", a[::-1, :].copy())\n    color_feat(\"mir_hv\", a[::-1, ::-1].copy())\n    if h == w:\n        color_feat(\"mir_t\", a.T.copy())\n        color_feat(\"mir_at\", a[::-1, ::-1].T.copy())\n    color_feat(\"row_col\", _line_color(a, bg, 1))\n    color_feat(\"col_col\", _line_color(a, bg, 0))\n\n    rr, cc = np.indices((h, w))\n    F[\"is_bg\"] = (a == bg).astype(np.int16)\n    for m in (2, 3):\n        F[f\"r%{m}\"] = (rr % m).astype(np.int16)\n        F[f\"c%{m}\"] = (cc % m).astype(np.int16)\n    F[\"r\"] = np.minimum(rr, 31).astype(np.int16)\n    F[\"c_idx\"] = np.minimum(cc, 31).astype(np.int16)\n    F[\"rb\"] = np.minimum(h - 1 - rr, 31).astype(np.int16)\n    F[\"cb\"] = np.minimum(w - 1 - cc, 31).astype(np.int16)\n    F[\"border\"] = ((rr == 0) | (cc == 0) | (rr == h - 1) | (cc == w - 1)).astype(np.int16)\n    F[\"diag\"] = ((rr == cc) | (rr + cc == w - 1)).astype(np.int16) if h == w else (rr == cc).astype(np.int16)\n\n    nonbg = (a != bg).astype(np.int16)\n    k8 = np.ones((3, 3), dtype=np.int16)\n    k8[1, 1] = 0\n    F[\"cnt8\"] = ndimage.convolve(nonbg, k8, mode=\"constant\").astype(np.int16)\n    k4 = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=np.int16)\n    F[\"cnt4\"] = ndimage.convolve(nonbg, k4, mode=\"constant\").astype(np.int16)\n    same4 = np.zeros((h, w), dtype=np.int16)\n    for dr, dc in DIRS4.values():\n        same4 += (_shift(a, dr, dc) == a).astype(np.int16)\n    F[\"same4\"] = same4\n\n    F[\"enclosed\"] = enclosed(a, bg).astype(np.int16)\n    # Between two cells of the same color horizontally / vertically.\n    rl, rr_ = F[\"ray_l\"], F[\"ray_r\"]\n    F[\"between_h\"] = np.where((rl == rr_) & (rl != NONE), rl, NONE).astype(np.int16)\n    ru, rd = F[\"ray_u\"], F[\"ray_d\"]\n    F[\"between_v\"] = np.where((ru == rd) & (ru != NONE), ru, NONE).astype(np.int16)\n    COLOR_FEATURES.update([\"between_h\", \"between_v\"])\n\n    # Object-level features (single-color 4-connected components).\n    for tag, diag, byc in ((\"o\", False, True), (\"m\", True, False)):\n        lab, sizes = label_map(a, bg, diag=diag, by_color=byc)\n        sizes = np.array(sizes)\n        size_map = sizes[lab]\n        F[tag + \"_size\"] = np.minimum(size_map, 31).astype(np.int16)\n        uniq = sorted(set(sizes[1:].tolist()))\n        if uniq:\n            rank_small = {s: i for i, s in enumerate(uniq)}\n            rank_large = {s: i for i, s in enumerate(reversed(uniq))}\n            F[tag + \"_rank_small\"] = np.vectorize(lambda s: min(rank_small.get(s, 30), 30) if s else NONE + 20)(size_map).astype(np.int16)\n            F[tag + \"_rank_large\"] = np.vectorize(lambda s: min(rank_large.get(s, 30), 30) if s else NONE + 20)(size_map).astype(np.int16)\n        else:\n            F[tag + \"_rank_small\"] = np.full((h, w), NONE + 20, dtype=np.int16)\n            F[tag + \"_rank_large\"] = np.full((h, w), NONE + 20, dtype=np.int16)\n        n = lab.max()\n        if n:\n            idx = np.arange(1, n + 1)\n            touch = np.zeros(n + 1, dtype=np.int16)\n            border_labels = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))\n            touch[border_labels] = 1\n            touch[0] = 0\n            F[tag + \"_touch\"] = touch[lab]\n            if not byc:\n                ncol = np.zeros(n + 1, dtype=np.int16)\n                mcol = np.full(n + 1, NONE, dtype=np.int16)\n                for i in idx:\n                    vals = a[lab == i]\n                    bc = np.bincount(vals, minlength=10)\n                    ncol[i] = (bc > 0).sum()\n                    # least-frequent color within the object (the \"marker\")\n                    nz = np.nonzero(bc)[0]\n                    mcol[i] = nz[np.argmin(bc[nz])]\n                F[tag + \"_ncol\"] = ncol[lab]\n                F[tag + \"_minor\"] = mcol[lab]\n                COLOR_FEATURES.add(tag + \"_minor\")\n        else:\n            F[tag + \"_touch\"] = np.zeros((h, w), dtype=np.int16)\n            if not byc:\n                F[tag + \"_ncol\"] = np.zeros((h, w), dtype=np.int16)\n                F[tag + \"_minor\"] = np.full((h, w), NONE, dtype=np.int16)\n                COLOR_FEATURES.add(tag + \"_minor\")\n\n    # Color frequency rank of own color (0 = most common).\n    counts = np.bincount(a.ravel(), minlength=11)\n    order = np.argsort(-counts, kind=\"stable\")\n    rank = np.empty(11, dtype=np.int16)\n    rank[order] = np.arange(11)\n    F[\"col_rank\"] = rank[a]\n    return F\n", "grid.py": "\"\"\"Grid utilities: background detection, connected components, geometry.\"\"\"\nfrom collections import Counter\n\nimport numpy as np\nfrom scipy import ndimage\n\nGrid = np.ndarray\n\n_STRUCT4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])\n_STRUCT8 = np.ones((3, 3), dtype=int)\n\n\ndef to_np(g):\n    return np.array(g, dtype=np.int8)\n\n\ndef to_list(a):\n    return [[int(v) for v in row] for row in a]\n\n\ndef background(a: Grid) -> int:\n    \"\"\"Most frequent color; ties broken toward 0.\"\"\"\n    vals, counts = np.unique(a, return_counts=True)\n    best = counts.max()\n    cands = vals[counts == best]\n    return 0 if 0 in cands else int(cands[0])\n\n\ndef components(a: Grid, bg: int, diag: bool = False, by_color: bool = True):\n    \"\"\"Connected components of non-background cells.\n\n    Returns list of dicts with mask, color(s), bbox, size.\n    \"\"\"\n    objs = []\n    struct = _STRUCT8 if diag else _STRUCT4\n    if by_color:\n        for c in np.unique(a):\n            if c == bg:\n                continue\n            lab, n = ndimage.label(a == c, structure=struct)\n            for i in range(1, n + 1):\n                objs.append(_obj(a, lab == i))\n    else:\n        lab, n = ndimage.label(a != bg, structure=struct)\n        for i in range(1, n + 1):\n            objs.append(_obj(a, lab == i))\n    return objs\n\n\ndef _obj(a, mask):\n    rows, cols = np.nonzero(mask)\n    r0, r1, c0, c1 = rows.min(), rows.max() + 1, cols.min(), cols.max() + 1\n    colors = Counter(a[mask].tolist())\n    return {\n        \"mask\": mask,\n        \"bbox\": (int(r0), int(r1), int(c0), int(c1)),\n        \"size\": int(mask.sum()),\n        \"color\": colors.most_common(1)[0][0],\n        \"ncolors\": len(colors),\n    }\n\n\ndef crop(a: Grid, bbox):\n    r0, r1, c0, c1 = bbox\n    return a[r0:r1, c0:c1]\n\n\ndef nonbg_bbox(a: Grid, bg: int):\n    rows, cols = np.nonzero(a != bg)\n    if len(rows) == 0:\n        return None\n    return (int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1)\n\n\ndef label_map(a: Grid, bg: int, diag: bool = False, by_color: bool = True):\n    \"\"\"Integer label per cell (0 = background) plus list of component sizes.\"\"\"\n    struct = _STRUCT8 if diag else _STRUCT4\n    out = np.zeros(a.shape, dtype=np.int32)\n    sizes = [0]\n    nxt = 1\n    if by_color:\n        for c in np.unique(a):\n            if c == bg:\n                continue\n            lab, n = ndimage.label(a == c, structure=struct)\n            for i in range(1, n + 1):\n                m = lab == i\n                out[m] = nxt\n                sizes.append(int(m.sum()))\n                nxt += 1\n    else:\n        lab, n = ndimage.label(a != bg, structure=struct)\n        out = lab.astype(np.int32)\n        sizes = [0] + [int((lab == i).sum()) for i in range(1, n + 1)]\n    return out, sizes\n\n\ndef enclosed(a: Grid, bg: int):\n    \"\"\"Mask of background cells not 4-connected to the border.\"\"\"\n    lab, n = ndimage.label(a == bg, structure=_STRUCT4)\n    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())\n    m = (lab > 0) & ~np.isin(lab, list(border))\n    return m\n\n\ndef separators(a: Grid):\n    \"\"\"Detect full rows/cols of a single color forming grid lines.\n\n    Returns (color, row_idx list, col_idx list) or None.\n    \"\"\"\n    h, w = a.shape\n    best = None\n    for c in np.unique(a):\n        rows = [r for r in range(h) if (a[r] == c).all()]\n        cols = [k for k in range(w) if (a[:, k] == c).all()]\n        if not rows and not cols:\n            continue\n        if len(rows) == h or len(cols) == w:\n            continue\n        score = len(rows) + len(cols)\n        if best is None or score > best[0]:\n            best = (score, int(c), rows, cols)\n    if best is None:\n        return None\n    return best[1], best[2], best[3]\n\n\ndef split_cells(a: Grid):\n    \"\"\"Split grid into sub-grids separated by separator lines.\"\"\"\n    sep = separators(a)\n    if sep is None:\n        return None\n    c, rows, cols = sep\n    h, w = a.shape\n\n    def spans(idx, n):\n        out, start = [], 0\n        for i in idx + [n]:\n            if i > start:\n                out.append((start, i))\n            start = i + 1\n        return out\n\n    rs, cs = spans(rows, h), spans(cols, w)\n    cells = [[a[r0:r1, c0:c1] for (c0, c1) in cs] for (r0, r1) in rs]\n    return c, cells\n\n\ndef mode_color(a: Grid, exclude=()):\n    cnt = Counter(a.ravel().tolist())\n    for c, _ in cnt.most_common():\n        if c not in exclude:\n            return c\n    return None\n", "rules.py": "\"\"\"MDL local-rule induction.\n\nGiven aligned (input-like grid, output grid) pairs of equal shape, search for a\nsmall set of per-cell features S and a target encoding E such that the lookup\ntable key(S) -> label(E) is consistent with every training cell. Each\nconsistent rule gets a description length; predictions from all consistent\nrules are pooled with weights exp(-DL) (a crude Bayesian model average).\n\"\"\"\nfrom itertools import combinations\n\nimport numpy as np\n\nfrom .features import COLOR_FEATURES, NONE, cell_features\n\nKEEP, COPY = 11, 12\nMAX_COMPRESSIVE_ENTRIES = 40\nMEMORIZATION_PENALTY = 1000.0\nBASE = 64\n\n\nclass RuleModel:\n    def __init__(self, names, subset, enc, table, cost, copy_feat=None):\n        self.names, self.subset, self.enc = names, subset, enc\n        self.table, self.cost, self.copy_feat = table, cost, copy_feat\n\n    def describe(self):\n        feats = [self.names[i] for i in self.subset]\n        return f\"rule[{self.enc}{':' + self.copy_feat if self.copy_feat else ''}] on {feats} ({len(self.table)} entries, DL={self.cost:.1f})\"\n\n\ndef _pack(F, subset):\n    key = np.zeros(F.shape[1], dtype=np.int64)\n    for i in subset:\n        key = key * BASE + F[i]\n    return key\n\n\ndef _impurity(key, y):\n    ky = key * 16 + y\n    u, cnt = np.unique(ky, return_counts=True)\n    k = u // 16\n    starts = np.flatnonzero(np.r_[True, k[1:] != k[:-1]])\n    return len(key) - np.maximum.reduceat(cnt, starts).sum(), len(starts)\n\n\ndef _feature_matrix(feat_dicts, names):\n    return np.stack([np.concatenate([fd[n].ravel() for fd in feat_dicts]) for n in names]).astype(np.int64)\n\n\ndef learn_rules(pairs, test_inputs, max_rules=40, beam=24, time_budget_feats=3):\n    \"\"\"pairs: list of (x, y) numpy grids with x.shape == y.shape.\n\n    Returns list of (prediction grids for test_inputs, log-weight, RuleModel).\n    \"\"\"\n    tr_feats = [cell_features(x) for x, _ in pairs]\n    te_feats = [cell_features(x) for x in test_inputs]\n    names = sorted(set.intersection(*[set(f) for f in tr_feats + te_feats]))\n    # Drop features that are constant across train and test (useless).\n    Ftr = _feature_matrix(tr_feats, names)\n    Fte = _feature_matrix(te_feats, names)\n    keep = [i for i in range(len(names)) if len(np.unique(np.r_[Ftr[i], Fte[i]])) > 1 or names[i] == \"c\"]\n    names = [names[i] for i in keep]\n    Ftr, Fte = Ftr[keep], Fte[keep]\n    ci = names.index(\"c\")\n    out = np.concatenate([y.ravel() for _, y in pairs]).astype(np.int64)\n    own = Ftr[ci]\n    changed = out != own\n\n    encodings = [(\"abs\", out, None, 0.0)]\n    if changed.any() and (~changed).any():\n        encodings.append((\"keep\", np.where(changed, out, KEEP), None, 0.0))\n        for j, n in enumerate(names):\n            if n in COLOR_FEATURES and n != \"c\" and (Ftr[j][changed] == out[changed]).all():\n                if len(np.unique(out[changed])) > 1:\n                    y = np.where(changed, COPY, KEEP)\n                    encodings.append((\"copy\", y, n, 1.0))\n    elif not changed.any():\n        return []\n\n    results = []\n    nf = len(names)\n    for enc, y, cf, enc_cost in encodings:\n        # Level 1 and 2: exhaustive; level 3: beam from best impure pairs.\n        scored = []\n        for i in range(nf):\n            scored.append(((i,), *_impurity(Ftr[i], y)))\n        pairs2 = []\n        for i, j in combinations(range(nf), 2):\n            imp, nk = _impurity(Ftr[i] * BASE + Ftr[j], y)\n            pairs2.append(((i, j), imp, nk))\n        scored += pairs2\n        if time_budget_feats >= 3:\n            best_pairs = sorted([p for p in pairs2 if p[1] > 0], key=lambda t: (t[1], t[2]))[:beam]\n            seen = set()\n            for (i, j), _, _ in best_pairs:\n                base = Ftr[i] * BASE + Ftr[j]\n                for k in range(nf):\n                    if k in (i, j):\n                        continue\n                    t = tuple(sorted((i, j, k)))\n                    if t in seen:\n                        continue\n                    seen.add(t)\n                    imp, nk = _impurity(base * BASE + Ftr[k], y)\n                    if imp == 0:\n                        scored.append((t, imp, nk))\n        for subset, imp, nk in scored:\n            if imp:\n                continue\n            m = _build(names, Ftr, Fte, y, subset, enc, cf, enc_cost, te_feats, test_inputs)\n            if m is not None:\n                results.append(m)\n    results.sort(key=lambda r: r[2].cost)\n    return results[:max_rules]\n\n\ndef _build(names, Ftr, Fte, y, subset, enc, cf, enc_cost, te_feats, test_inputs):\n    ktr = _pack(Ftr, subset)\n    kte = _pack(Fte, subset)\n    table = dict(zip(ktr.tolist(), y.tolist()))\n    labels = np.array([table.get(k, -1) for k in kte.tolist()], dtype=np.int64)\n    unseen = labels < 0\n    n_unseen_keys = len(set(kte[unseen].tolist()))\n    if unseen.any() and enc == \"abs\":\n        return None  # no sensible default for absolute encodings\n    labels[unseen] = KEEP\n    # Description length: table entries + feature choice + unseen penalty.\n    cost = len(table) * 1.0 + 1.5 * len(subset) + enc_cost + 4.0 * n_unseen_keys\n    if len(table) > MAX_COMPRESSIVE_ENTRIES:\n        cost += MEMORIZATION_PENALTY  # fails the compression test: keep only as last resort\n    # Reward rules whose training support is large relative to table size.\n    own = Fte[names.index(\"c\")]\n    pred = np.where(labels == KEEP, own, labels)\n    if enc == \"copy\":\n        src = Fte[names.index(cf)]\n        pred = np.where(labels == COPY, src, pred)\n    if (pred >= 10).any():\n        return None\n    grids, off = [], 0\n    for x in test_inputs:\n        n = x.size\n        grids.append(pred[off:off + n].reshape(x.shape).astype(np.int8))\n        off += n\n    return grids, -cost / 2.0, RuleModel(names, subset, enc, table, cost, cf)\n", "runner.py": "\"\"\"Parallel batch runner used by the Kaggle notebook.\"\"\"\nimport json\nimport os\nimport signal\nimport time\nfrom multiprocessing import Pool\n\nfrom .solver import solve_task\n\n\ndef _alarm(*_):\n    raise TimeoutError()\n\n\ndef _run(args):\n    tid, task, per_task = args\n    signal.signal(signal.SIGALRM, _alarm)\n    signal.alarm(int(per_task * 2) + 5)\n    try:\n        out, info = solve_task(task, time_limit=per_task)\n    except BaseException as e:  # never lose a task: fall back to identity\n        out, info = None, [repr(e)]\n    finally:\n        signal.alarm(0)\n    if out is None:\n        out = [{\"attempt_1\": t[\"input\"], \"attempt_2\": t[\"input\"]} for t in task[\"test\"]]\n    return tid, out, info\n\n\ndef solve_file(path, out_path=\"submission.json\", per_task=60, jobs=None):\n    tasks = json.load(open(path))\n    t0 = time.time()\n    with Pool(jobs or os.cpu_count()) as p:\n        results = p.map(_run, [(tid, t, per_task) for tid, t in tasks.items()], chunksize=1)\n    submission = {tid: out for tid, out, _ in results}\n    json.dump(submission, open(out_path, \"w\"))\n    print(f\"wrote {out_path}: {len(submission)} tasks in {time.time() - t0:.0f}s\")\n    return submission, {tid: info for tid, _, info in results}\n", "solver.py": "\"\"\"Top-level solver: transform search + MDL local-rule induction + model averaging.\"\"\"\nimport time\nfrom collections import defaultdict\n\nimport numpy as np\n\nfrom . import dsl\nfrom .grid import to_list, to_np\nfrom .rules import learn_rules\n\nGEOM = [t for t in dsl.TRANSFORMS if t[0] in (\n    \"rot90\", \"rot180\", \"rot270\", \"flipud\", \"fliplr\", \"transpose\", \"antitranspose\", \"crop_nonbg\",\n    \"dedupe_rc\", \"drop_bg_lines\", \"remove_seps\")]\n\n\ndef _key(grids):\n    return tuple((g.shape, g.tobytes()) for g in grids)\n\n\ndef _apply_all(fn, grids):\n    out = []\n    for g in grids:\n        r = dsl.apply(fn, g)\n        if r is None:\n            return None\n        out.append(r)\n    return out\n\n\ndef solve_task(task, time_limit=30.0, max_rule_transforms=6, verbose=False):\n    t0 = time.time()\n    tr_in = [to_np(p[\"input\"]) for p in task[\"train\"]]\n    tr_out = [to_np(p[\"output\"]) for p in task[\"train\"]]\n    te_in = [to_np(p[\"input\"]) for p in task[\"test\"]]\n\n    votes = defaultdict(lambda: -np.inf)\n    preds = {}\n    explain = {}\n\n    def add(grids, logw, why):\n        k = _key(grids)\n        votes[k] = np.logaddexp(votes[k], logw)\n        preds[k] = grids\n        if k not in explain or logw > explain[k][0]:\n            explain[k] = (logw, why)\n\n    # 1. Single transforms (exact, or as a basis for rule learning).\n    shape_ok = []\n    seen_inter = set()\n    for name, fn, cost in dsl.TRANSFORMS:\n        xs = _apply_all(fn, tr_in)\n        if xs is None or any(x.shape != y.shape for x, y in zip(xs, tr_out)):\n            continue\n        ts = _apply_all(fn, te_in)\n        if ts is None:\n            continue\n        if all((x == y).all() for x, y in zip(xs, tr_out)):\n            add(ts, -cost, f\"exact:{name}\")\n            continue\n        k = _key(xs) + _key(ts)\n        if k in seen_inter:\n            continue\n        seen_inter.add(k)\n        acc = np.mean([(x == y).mean() for x, y in zip(xs, tr_out)])\n        shape_ok.append((-(acc), cost, name, xs, ts))\n\n    # 2. Depth-2 exact compositions (cheap check only).\n    if not votes:\n        for n1, f1, c1 in dsl.TRANSFORMS:\n            if time.time() - t0 > time_limit * 0.3:\n                break\n            xs1 = _apply_all(f1, tr_in)\n            if xs1 is None or n1 == \"id\":\n                continue\n            for n2, f2, c2 in GEOM:\n                xs = _apply_all(f2, xs1)\n                if xs is None or any(x.shape != y.shape for x, y in zip(xs, tr_out)):\n                    continue\n                if all((x == y).all() for x, y in zip(xs, tr_out)):\n                    ts1 = _apply_all(f1, te_in)\n                    ts = ts1 and _apply_all(f2, ts1)\n                    if ts:\n                        add(ts, -(c1 + c2), f\"exact:{n1}>{n2}\")\n\n    # 3. MDL rule induction on top of shape-preserving transforms.\n    shape_ok.sort(key=lambda t: (t[0], t[1]))\n    for negacc, cost, name, xs, ts in shape_ok[:max_rule_transforms]:\n        if time.time() - t0 > time_limit:\n            break\n        try:\n            rules = learn_rules(list(zip(xs, tr_out)), ts)\n        except Exception as e:  # keep going on odd inputs\n            if verbose:\n                print(\"rule error\", name, e)\n            continue\n        for grids, w, model in rules:\n            add(grids, w - cost, f\"{name}+{model.describe()}\")\n\n    ranked = sorted(votes, key=lambda k: -votes[k])\n    attempts = [preds[k] for k in ranked[:2]]\n    info = [explain[k][1] for k in ranked[:2]]\n    while len(attempts) < 2:\n        attempts.append(te_in)\n        info.append(\"fallback:identity\")\n    out = []\n    for i in range(len(te_in)):\n        out.append({\"attempt_1\": to_list(attempts[0][i]), \"attempt_2\": to_list(attempts[1][i])})\n    return out, info\n"}
for k, v in FILES.items():
    open(os.path.join('arcsolver', k), 'w').write(v)
print(sorted(FILES))

In [ ]:
import glob, json
from arcsolver.runner import solve_file

cands = glob.glob('/kaggle/input/**/arc-agi_test_challenges.json', recursive=True)
TEST = cands[0] if cands else 'arc-agi_test_challenges.json'
print('input:', TEST)
submission, info = solve_file(TEST, 'submission.json', per_task=60)


In [ ]:
# Local sanity check when the public evaluation solutions are available.
sol = glob.glob('/kaggle/input/**/arc-agi_evaluation_solutions.json', recursive=True)
if sol and 'evaluation' in TEST:
    sols = json.load(open(sol[0]))
    score = 0
    for tid, outs in submission.items():
        s = sum(o['attempt_1'] == g or o['attempt_2'] == g for o, g in zip(outs, sols[tid]))
        score += s / len(outs)
    print('score', score, '/', len(submission))
